In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import spatialdata as sd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats



In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})


plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
all_cells_adata = sc.read_h5ad(str(P.processed.adata.all_cells / "xenium_allcells_combo_all_cells_16_50_harmonized_clustered.h5ad"))

In [ ]:
core_anno = pd.read_csv(str(P.metadata / "core_annotations_final.csv"))

In [ ]:
demo_cols = {"age": "Age", "gender": "Gender", "colon_location": "Colon Location"}
demo = (
    core_anno[["patient_id"] + list(demo_cols.keys())]
    .rename(columns=demo_cols)
    .drop_duplicates("patient_id")
)

original_index = all_cells_adata.obs.index
all_cells_adata.obs = all_cells_adata.obs.merge(demo, on="patient_id", how="left")
all_cells_adata.obs.index = original_index

In [ ]:
all_cells_adata_uniform = all_cells_adata.copy()

mixed_cores_list = core_anno.loc[core_anno["mixed_core_tissue_type"].fillna("").astype(str).str.upper() == "TRUE", "core_id"].tolist()

def label_core(row):
    if row["core_id"] in mixed_cores_list:
        return "Mixed"
    if row["distant_normal"] == True:
        return "Dist_N"
    return row["tissue_type"]

all_cells_adata_uniform.obs["Tissue type"] = all_cells_adata_uniform.obs.apply(label_core, axis=1)

print(all_cells_adata_uniform.obs["Tissue type"].value_counts())

In [ ]:
cell_ids = all_cells_adata_uniform.obs["cell_id"].astype(str)
mask = ~cell_ids.duplicated(keep="first")
all_cells_adata_uniform = all_cells_adata_uniform[mask].copy()

print(f"Cells before: {len(mask)}")
print(f"Cells after:  {all_cells_adata_uniform.n_obs}")
print(f"Removed:      {(~mask).sum()}")

In [ ]:
core_info = (
    all_cells_adata_uniform.obs
    .drop_duplicates("core_id")
    [["core_id", "patient_id", "Tissue type"]]
)

core_counts = (
    core_info
    .groupby(["patient_id", "Tissue type"])
    .size()
    .unstack(fill_value=0)
)

col_map = {
    "Dist_N": "# Dist_N cores",
    "Adj_N": "# Adj_N cores",
    "AD": "# AD cores",
    "CA": "# CA cores",
    "Mixed": "# Mixed cores",
}
core_counts = core_counts.rename(columns=col_map)
for col in col_map.values():
    if col not in core_counts.columns:
        core_counts[col] = 0
core_counts["# Total cores"] = core_counts[list(col_map.values())].sum(axis=1)

tissue_order_map = {"Adj_N": 0, "AD": 1, "CA": 2}

mixed_anno = core_anno[core_anno["mix_type"].notna() & (core_anno["mix_type"] != "")].copy()

def mix_type_to_tissue(val):
    val = val.replace("LGD", "AD").replace("HGD", "AD")
    parts = [p.strip() for p in val.split(",")]
    parts = list(dict.fromkeys(parts))
    parts = sorted(parts, key=lambda x: tissue_order_map.get(x, 99))
    return "/".join(parts)

mixed_anno["composition"] = mixed_anno["mix_type"].apply(mix_type_to_tissue)

mixed_per_patient = (
    mixed_anno
    .sort_values("core_id")
    .groupby("patient_id")["composition"]
    .apply(lambda x: ", ".join(x))
    .rename("Mixed core composition")
)

patient_summary = (
    demo[["patient_id", "Age", "Gender", "Colon Location"]]
    .merge(core_counts, left_on="patient_id", right_index=True, how="inner")
    .merge(mixed_per_patient, left_on="patient_id", right_index=True, how="left")
)

patient_summary["Mixed core composition"] = patient_summary["Mixed core composition"].fillna("")

patient_summary = patient_summary[[
    "patient_id", "Age", "Gender", "Colon Location",
    "# Total cores", "# Separate N cores", "# Adj_N cores",
    "# AD cores", "# CA cores", "# Mixed cores", "Mixed core composition"
]].rename(columns={"patient_id": "Patient"}).sort_values("Patient").reset_index(drop=True)

patient_summary

In [ ]:
print(patient_summary)

In [ ]:
patient_summary.to_csv(str(P.metadata / "pt-summary" / "patient_summary.csv"))

In [ ]:
fig_outdir = str(P.results.figures / "supplemental-tables")

In [ ]:
patient_summary['Colon Location'] = patient_summary['Colon Location'].replace('Sigmoid Colon', 'Sigmoid')

import matplotlib.pyplot as plt

header_color = "#030506"
alt_color_1 = '#f8f9fa'
alt_color_2 = 'white'
edge_color = '#dee2e6'
font_size = 9

fig1, ax1 = plt.subplots(figsize=(12, 6))
ax1.axis('off')

col_labels_1 = patient_summary.columns.tolist()
cell_text_1 = patient_summary.astype(str).values.tolist()

tbl1 = ax1.table(
    cellText=cell_text_1,
    colLabels=col_labels_1,
    loc='center',
    cellLoc='center',
)

tbl1.auto_set_font_size(False)
tbl1.set_fontsize(font_size)
tbl1.scale(1, 1.5)

for j in range(len(col_labels_1)):
    cell = tbl1[0, j]
    cell.set_facecolor(header_color)
    cell.set_text_props(color='white', fontweight='bold', fontsize=font_size)
    cell.set_edgecolor('white')

for i in range(len(cell_text_1)):
    for j in range(len(col_labels_1)):
        cell = tbl1[i + 1, j]
        cell.set_edgecolor(edge_color)
        cell.set_facecolor(alt_color_1 if i % 2 == 0 else alt_color_2)

tbl1.auto_set_column_width(col=list(range(len(col_labels_1))))
plt.title('Patient and Core Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(fig_outdir, 'patient_summary_table.pdf'), bbox_inches='tight', dpi=300)
plt.show()

n = len(patient_summary)

age = patient_summary['Age']
age_median = age.median()
age_q1 = age.quantile(0.25)
age_q3 = age.quantile(0.75)

gender_counts = patient_summary['Gender'].value_counts()
location_counts = patient_summary['Colon Location'].value_counts().sort_index()

display_rows = [
    ['n', str(n)],
    ['Age, median [Q1, Q3]', f'{age_median:.1f} [{age_q1:.1f}, {age_q3:.1f}]'],
    ['Gender, n (%)', ''],
]
for val, count in gender_counts.items():
    display_rows.append([f'   {val}', f'{count} ({count/n*100:.1f})'])

display_rows.append(['Colon Location, n (%)', ''])
for val, count in location_counts.items():
    display_rows.append([f'   {val}', f'{count} ({count/n*100:.1f})'])

fig2, ax2 = plt.subplots(figsize=(6, len(display_rows) * 0.3 + 1))
ax2.axis('off')

col_labels_2 = ['', f'Overall (n={n})']

tbl2 = ax2.table(
    cellText=display_rows,
    colLabels=col_labels_2,
    loc='center',
    cellLoc='left',
)

tbl2.auto_set_font_size(False)
tbl2.set_fontsize(font_size + 1)
tbl2.scale(1, 1.5)

for j in range(len(col_labels_2)):
    cell = tbl2[0, j]
    cell.set_facecolor(header_color)
    cell.set_text_props(color='white', fontweight='bold', fontsize=font_size + 1)
    cell.set_edgecolor('white')

for i in range(len(display_rows)):
    for j in range(len(col_labels_2)):
        cell = tbl2[i + 1, j]
        cell.set_edgecolor(edge_color)
        cell.set_facecolor(alt_color_1 if i % 2 == 0 else alt_color_2)
        if j == 0 and not display_rows[i][0].startswith(' '):
            cell.set_text_props(fontweight='bold')

tbl2.auto_set_column_width(col=list(range(len(col_labels_2))))
plt.title('Patient Cohort Characteristics', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(fig_outdir, 'table1_cohort_characteristics.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
print(f"Total cores: {patient_summary['# Total cores'].sum()}")

In [ ]:
n = len(patient_summary)

# Age
age = patient_summary['Age']
age_median = age.median()
age_q1 = age.quantile(0.25)
age_q3 = age.quantile(0.75)

# Gender
gender_counts = patient_summary['Gender'].value_counts()

# Colon Location
location_counts = patient_summary['Colon Location'].value_counts().sort_index()

display_rows = [
    ['n', str(n)],
    ['Age, median [Q1, Q3]', f'{age_median:.1f} [{age_q1:.1f}, {age_q3:.1f}]'],
    ['Gender, n (%)', ''],
]
for val, count in gender_counts.items():
    display_rows.append([f'   {val}', f'{count} ({count/n*100:.1f})'])

display_rows.append(['Colon Location, n (%)', ''])
for val, count in location_counts.items():
    display_rows.append([f'   {val}', f'{count} ({count/n*100:.1f})'])

# Render
fig, ax = plt.subplots(figsize=(8, len(display_rows) * 0.4 + 1))
ax.axis('off')

col_labels = ['', f'Overall (n={n})']

tbl = ax.table(
    cellText=display_rows,
    colLabels=col_labels,
    loc='center',
    cellLoc='left',
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)

for j in range(len(col_labels)):
    cell = tbl[0, j]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=10)
    cell.set_edgecolor('white')

for i in range(len(display_rows)):
    for j in range(len(col_labels)):
        cell = tbl[i + 1, j]
        cell.set_edgecolor('#dee2e6')
        cell.set_facecolor('#f8f9fa' if i % 2 == 0 else 'white')
        if j == 0 and not display_rows[i][0].startswith(' '):
            cell.set_text_props(fontweight='bold')

tbl.auto_set_column_width(col=list(range(len(col_labels))))

plt.title('Table 1: Patient Cohort Characteristics', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(fig_outdir, 'table1_cohort_characteristics.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.transforms import Bbox

PCA_AX_WIDTH = 7
PCA_AX_HEIGHT = 6


def compute_core_pca(adata, n_components=10):
    """
    Compute PCA where each dot is one core.

    Returns
    -------
    coords     : np.ndarray, shape (n_cores, n_components)
    core_meta  : pd.DataFrame, indexed by core_id (all obs metadata, one row per core)
    pca        : fitted PCA object
    """
    core_expr = (
        adata.to_df()
        .assign(core_id=adata.obs["core_id"].values)
        .groupby("core_id")
        .mean()
    )

    core_meta = (
        adata.obs
        .drop_duplicates("core_id")
        .set_index("core_id")
        .loc[core_expr.index]
    )

    # Run PCA
    pca = PCA(n_components=n_components)
    coords = pca.fit_transform(core_expr.values)

    return coords, core_meta, pca


def plot_core_pca(coords, core_meta, pca, color_by, palette=None,
                  title=None, save=None, label_cores=False):
    """
    Plot pre-computed core PCA, colored by a metadata column.
    The axes area is fixed at PCA_AX_WIDTH x PCA_AX_HEIGHT inches;
    the legend sits outside and the figure expands to fit it.
    """
    categories = core_meta[color_by].astype("category")

    if palette is None:
        all_colors = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors) + list(plt.cm.tab20c.colors)
        palette = {cat: all_colors[i] for i, cat in enumerate(categories.cat.categories)}

    fig = plt.figure()
    ax = fig.add_axes([0, 0, 1, 1])  

    for cat in categories.cat.categories:
        mask = (categories == cat).values
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=100, alpha=0.8, label=cat, color=palette[cat])
        if label_cores:
            for idx, (core_id, _) in enumerate(core_meta[categories == cat].iterrows()):
                ax.annotate(core_id, (coords[mask][idx, 0], coords[mask][idx, 1]),
                            fontsize=7, xytext=(4, 4), textcoords="offset points")

    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    ax.set_xlabel(f"PC1 ({var_pc1:.1f}%)", fontsize=12)
    ax.set_ylabel(f"PC2 ({var_pc2:.1f}%)", fontsize=12)
    ax.set_title(title or f"Per-core PCA colored by {color_by}", fontsize=14)
    ax.legend(markerscale=1, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")

    fig.set_size_inches(PCA_AX_WIDTH + 3, PCA_AX_HEIGHT)  
    ax.set_position([0.1, 0.1, PCA_AX_WIDTH / (PCA_AX_WIDTH + 3) * 0.85, 0.8])

    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
coords, core_meta, pca = compute_core_pca(all_cells_adata_uniform)


In [ ]:
all_cells_adata_uniform.obs['Colon Location'] = all_cells_adata_uniform.obs['Colon Location'].replace('Sigmoid Colon', 'Sigmoid')

In [ ]:
core_meta = (
    all_cells_adata_uniform.obs
    .drop_duplicates("core_id")
    .set_index("core_id")
    .loc[core_meta.index]
)


In [ ]:
import os
os.makedirs(str(P.results.figures / 'supp_8'), exist_ok=True)

plot_core_pca(coords, core_meta, pca, color_by='batch', save=str(P.results.figures / 'supplemental-pca' / 'pca_batch.pdf'))


In [ ]:
palette = {
    "Dist_N": "green",
    "AD":         "gold",
    "CA":         "crimson",
    "Adj_N":          "steelblue",
    "Mixed":      "orchid",
}

plot_core_pca(coords, core_meta, pca, color_by="patient_id", palette=None, label_cores=False, save=str(P.results.figures / 'supplemental-pca' / 'pca_patient_id.pdf'))
plot_core_pca(coords, core_meta, pca, color_by="Tissue type", palette=palette, label_cores=False, save=str(P.results.figures / 'supplemental-pca' / 'pca_tissue_type.pdf'))
plot_core_pca(coords, core_meta, pca, color_by="Colon Location", palette=None, label_cores=False, save=str(P.results.figures / 'supplemental-pca' / 'pca_colon_location.pdf'))


In [ ]:
gender_palette = {"M": "steelblue", "F": "crimson"}
plot_core_pca(coords, core_meta, pca, color_by="Gender", palette=gender_palette, label_cores=False, save=str(P.results.figures / 'supplemental-pca' / 'pca_gender.pdf'))

def plot_core_pca_continuous(adata, color_by, cmap="viridis", n_components=10,
                              title=None, save=None, label_cores=True):
    core_expr = (
        adata.to_df()
        .assign(core_id=adata.obs["core_id"].values)
        .groupby("core_id")
        .mean()
    )

    core_meta = (
        adata.obs[["core_id", color_by]]
        .drop_duplicates("core_id")
        .set_index("core_id")
        .loc[core_expr.index]
    )

    pca = PCA(n_components=n_components)
    coords = pca.fit_transform(core_expr.values)
    values = core_meta[color_by].astype(float).values

    fig = plt.figure()
    fig.set_size_inches(PCA_AX_WIDTH + 3, PCA_AX_HEIGHT)
    ax_pos = [0.1, 0.1, PCA_AX_WIDTH / (PCA_AX_WIDTH + 3) * 0.85, 0.8]
    ax = fig.add_axes(ax_pos)

    sc = ax.scatter(coords[:, 0], coords[:, 1],
                c=values, cmap=cmap, s=100, alpha=0.8, rasterized=True)

    cbar_ax = fig.add_axes([ax_pos[0] + ax_pos[2] + 0.02, ax_pos[1], 0.02, ax_pos[3]])
    fig.colorbar(sc, cax=cbar_ax, label=color_by)

    if label_cores:
        for idx, core_id in enumerate(core_meta.index):
            ax.annotate(core_id, (coords[idx, 0], coords[idx, 1]),
                        fontsize=7, xytext=(4, 4), textcoords="offset points")

    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    ax.set_xlabel(f"PC1 ({var_pc1:.1f}%)", fontsize=12)
    ax.set_ylabel(f"PC2 ({var_pc2:.1f}%)", fontsize=12)
    ax.set_title(title or f"Per-core PCA colored by {color_by}", fontsize=14)

    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()

plot_core_pca_continuous(all_cells_adata_uniform, color_by="Age", cmap="plasma", label_cores=False, save=str(P.results.figures / 'supplemental-pca' / 'pca_age.pdf'))
